# 2026 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pytz
import seaborn as sns

from usagestats import visual
from usagestats.bookingdb import BookingDB

folder = Path('/run/user/1000/gvfs/smb-share:server=istore,share=microscopy_group_share,user=jeromeb/Admin/Usage Stats/2026')
db_file = folder / 'bookings.db'
local_tz = pytz.timezone("Europe/London")
start_date = datetime.fromisoformat('2025-11-04').replace(tzinfo=local_tz)
num_weeks = 52
end_date = start_date + timedelta(weeks=num_weeks)  # iso-format data yyyy-mm-dd
drop_bookings_type = ["service", "maintenance", "training"]
drop_instrument = ["Analysis PC1", "Analysis PC2", "Analysis PC3", "Analysis PC4"]

## Create the database

In [ ]:
booking_types = ['default','service', 'maintenance', 'training']
divisions = ['Unknown','Cell Biology','Neurobiology','PNAC','Structural Studies','University']
groups = pd.read_csv(folder/'groups.csv')
users = pd.read_csv(folder/'users.csv')
training = pd.read_csv(folder/'training.csv')
calendars = pd.read_csv(folder/'calendars.csv')

# get a cookie from the curl command copied from firefox dev tools and save it in a cookie.txt file
with open('cookie.txt','r') as f:
    curl_str = f.read()

print(f"there are {len(calendars)} instruments")

# check that the calendars url are uniques
assert calendars['url'].is_unique, "Calendar URLs are not unique!"

training = training[~training["Email"].isna()]
training = training[~training["Group"].isna()]
training["Year"] = training["Date Requested"].apply(lambda x:str(x).split("/")[-1])
training["email"] = training["Email"].apply(lambda x:x.split("@")[0])
training["group"] = training["Group"].apply(lambda x:x.split(" ")[-1])
training["user"] = training["Name"]
training[["user","group","email"]]

users = pd.concat((training[["user","group","email"]],users[["user","group","email"]])).drop_duplicates()

# Load all calendars in the database
Path(db_file).unlink(True)
with BookingDB(db_file, curl_str=curl_str) as db:
    db.create_calendars_from_df(calendars)
    db.create_instruments()
    db.create_booking_types(booking_types)
    db.create_division(divisions)
    db.create_groups(groups)
    df, duplicates = db.create_events(users)

there are 31 instruments
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constraint failed: calendars.url
UNIQUE constrai

In [ ]:

with BookingDB(db_file) as db:
    bookings = db.get_events_in_range(start_date, end_date)

newusers = bookings[bookings['group']=='Unknown'][['email', 'user','group','division']].drop_duplicates().reset_index()
print(f"Number of new users: {len(newusers)}")

# save the list to a csv file
new_users_path = folder/'new users.csv'

if not new_users_path.exists():
    newusers.to_csv(folder/'new users.csv')
else:
    print(f"File '{new_users_path.name}' exists.")
newusers